In [80]:
import os
import zipfile
import geopandas as gpd

In [67]:
# URL cartes.gouv
url_wfs = "https://data.geopf.fr/wfs?SERVICE=WFS&REQUEST=GetFeature&VERSION=2.0.0&TYPENAMES=AOC-VITICOLES%3Aaire_parcellaire&OUTPUTFORMAT=application%2Fjson&SRSNAME=EPSG%3A4326"

# Lecture directe du flux JSON/GeoJSON
gdf = gpd.read_file(url_wfs)


In [82]:
# 2. CHARGEMENT DES RÉGIONS DU REPERTOIRE OFFICIEL (CORRIGÉ)
print("⏳ Chargement du fond de carte officiel des Régions de France...")
# Utilisation de l'URL brute corrigée et simplifiée
url_regions_valide = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions-version-simplifiee.geojson"
france_regions = gpd.read_file(url_regions_valide)

# On filtre les codes INSEE pour exclure les DOM-TOM (les DOM ont des codes à 2 chiffres spécifiques)
# Les 13 régions métropolitaines restent ainsi seules dans notre jeu de données
regions_metropole = france_regions[~france_regions['code'].isin(['01', '02', '03', '04', '06'])]

# 3. PROJECTION ET HARMONISATION GÉOGRAPHIQUE (WGS 84 - EPSG:4326)
# Alignement obligatoire des données sur la projection nationale
france_projected = regions_metropole.to_crs(epsg=4326)
gdf_projected = gdf.to_crs(epsg=4326)

france_projected = france_projected.assign(type_obj='Region')
gdf_projected = gdf_projected.assign(type_obj='Appellatio')

# 4. EXPORTATION EN DEUX FICHIERS DISTINCTS
dossier_raw = r"C:\Users\Utilisateur\Desktop\Formation Data\Projet Wine Cellar\data\raw"
os.makedirs(dossier_raw, exist_ok=True)

# Noms de base pour les exports
nom_regions = "fond_regions"
nom_appellations = "mes_appellations"

# Export temporaire des fichiers Shapefile individuels
france_projected[['type_obj', 'nom', 'geometry']].to_file(os.path.join(dossier_raw, f"{nom_regions}.shp"), driver="ESRI Shapefile", encoding="utf-8")
gdf_projected.to_file(os.path.join(dossier_raw, f"{nom_appellations}.shp"), driver="ESRI Shapefile", encoding="utf-8")

# Compression en deux dossiers.zip
def zipper_shapefile(dossier, nom_base):
    chemin_zip = os.path.join(dossier, f"{nom_base}.zip")
    extensions = ['.shp', '.shx', '.dbf', '.prj', '.cpg']
    
    with zipfile.ZipFile(chemin_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for ext in extensions:
            fichier_composant = f"{nom_base}{ext}"
            chemin_composant = os.path.join(dossier, fichier_composant)
            if os.path.exists(chemin_composant):
                # Ajoute le fichier dans le ZIP et le supprime du dossier pour laisser propre
                zipf.write(chemin_composant, fichier_composant)
                os.remove(chemin_composant)
    return chemin_zip

print("⏳ Compression des fichiers en cours...")
zip1 = zipper_shapefile(dossier_raw, nom_regions)
zip2 = zipper_shapefile(dossier_raw, nom_appellations)

print("🎉 EXPORTS ET COMPRESSIONS TERMINÉS AVEC SUCCÈS !")
print(f"📦 Archive 1 : {zip1}")
print(f"📦 Archive 2 : {zip2}")
print("💡 Glissez simplement ces 2 fichiers .zip ensemble dans Mapshaper !")




⏳ Chargement du fond de carte officiel des Régions de France...
⏳ Compression des fichiers en cours...
🎉 EXPORTS ET COMPRESSIONS TERMINÉS AVEC SUCCÈS !
📦 Archive 1 : C:\Users\Utilisateur\Desktop\Formation Data\Projet Wine Cellar\data\raw\fond_regions.zip
📦 Archive 2 : C:\Users\Utilisateur\Desktop\Formation Data\Projet Wine Cellar\data\raw\mes_appellations.zip
💡 Glissez simplement ces 2 fichiers .zip ensemble dans Mapshaper !


In [70]:
print(dossier_raw)

C:\Users\Utilisateur\Desktop\Formation Data\Projet Wine Cellar\data\raw
